<a href="https://colab.research.google.com/github/navadiyajils007-pixel/aiml_assignment/blob/main/Prml_assignment1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
REPETITIONS = [100, 500, 1000, 5000, 10000, 50000, 100000]
MAX_ROLLS = 50

out = Path("q1_output")
hist_root = out / "histograms"
hist_root.mkdir(parents=True, exist_ok=True)
rng = np.random.default_rng(SEED)
rows = []

for reps in REPETITIONS:
    rolls = rng.integers(1, 7, size=(reps, MAX_ROLLS), dtype=np.int16)
    cumulative_sums = np.cumsum(rolls, axis=1)

    for n in range(1, MAX_ROLLS + 1):
        values = cumulative_sums[:, n - 1]
        empirical_mean = float(np.mean(values))
        empirical_variance = float(np.var(values, ddof=0))
        theoretical_mean = 3.5 * n
        theoretical_variance = (35 / 12) * n

        rows.append({
            "number_of_rolls": n,
            "repetitions": reps,
            "empirical_mean": empirical_mean,
            "empirical_variance": empirical_variance,
            "theoretical_mean": theoretical_mean,
            "theoretical_variance": theoretical_variance
        })

        roll_dir = hist_root / f"rolls_{n:02d}"
        roll_dir.mkdir(exist_ok=True)
        bins = np.arange(values.min() - 0.5, values.max() + 1.5, 1)

        plt.figure(figsize=(8, 5))
        plt.hist(values, bins=bins, edgecolor="black")
        plt.title(f"Sum of {n} Die Roll(s), {reps} Experiments")
        plt.xlabel("Observed value" if n == 1 else "Sum")
        plt.ylabel("Frequency")
        plt.tight_layout()
        plt.savefig(roll_dir / f"repetitions_{reps}.png", dpi=140)
        plt.close()

result = pd.DataFrame(rows)
result.to_csv(out / "mean_variance.csv", index=False)
print(result.to_string(index=False))
print(f"\nSaved {len(result)} histogram cases and mean/variance table in {out.resolve()}")


In [ ]:
from pathlib import Path
from math import comb
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
n = 100
observed_positive = 97
success_rates = [0.90, 0.93, 0.95, 0.97, 0.99]
simulations = 100000
out = Path("q2_output")
out.mkdir(exist_ok=True)
rng = np.random.default_rng(SEED)
rows = []

for p in success_rates:
    probability = comb(n, observed_positive) * (p ** observed_positive) * ((1 - p) ** (n - observed_positive))
    rows.append({"success_rate": p, "P(X=97)": probability})

    samples = rng.binomial(n=n, p=p, size=simulations)
    bins = np.arange(samples.min() - 0.5, samples.max() + 1.5, 1)
    plt.figure(figsize=(8, 5))
    plt.hist(samples, bins=bins, edgecolor="black")
    plt.axvline(observed_positive, linestyle="--", linewidth=2, label="Observed = 97")
    plt.title(f"Binomial(n=100, p={p:.2f})")
    plt.xlabel("Number of Positive Reviews")
    plt.ylabel("Frequency")
    plt.legend()
    plt.tight_layout()
    plt.savefig(out / f"binomial_p_{p:.2f}.png", dpi=140)
    plt.close()

result = pd.DataFrame(rows)
result.to_csv(out / "candidate_likelihoods.csv", index=False)
print(result.to_string(index=False))
print(f"\nEstimated success rate = {observed_positive / n:.2f} = 97%")


In [ ]:
import numpy as np
import pandas as pd

def entropy_from_counts(counts):
    p = counts / counts.sum()
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

df = pd.read_csv("advertising.csv")
bins = 10
rows = []

for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]) and df[col].nunique() > bins:
        binned = pd.cut(df[col], bins=bins, include_lowest=True)
        counts = binned.value_counts(sort=False)
        method = "10 equal-width bins"
    else:
        counts = df[col].value_counts(dropna=False)
        method = "categorical frequencies"

    rows.append({
        "column": col,
        "entropy_bits": entropy_from_counts(counts),
        "method": method
    })

result = pd.DataFrame(rows)
print(result.to_string(index=False))
result.to_csv("q3_entropy_results.csv", index=False)


In [ ]:
import pandas as pd

df = pd.read_csv("enjoy_sport.csv")
attributes = list(df.columns[:-1])
target = df.columns[-1]
hypothesis = [None] * len(attributes)

print("Find-S updates:")
for i, row in df.iterrows():
    if str(row[target]).strip().lower() == "yes":
        x = [row[a] for a in attributes]
        if all(v is None for v in hypothesis):
            hypothesis = x.copy()
        else:
            for j, value in enumerate(x):
                if hypothesis[j] != value:
                    hypothesis[j] = "?"
        print(f"After positive example {i + 1}: {tuple(hypothesis)}")

print("\nMaximally Specific Hypothesis:")
print(tuple(hypothesis))


In [ ]:
import pandas as pd

def covers(h, x):
    return all(hv is not None and (hv == "?" or hv == xv) for hv, xv in zip(h, x))

def more_general_or_equal(h1, h2):
    return all(a == "?" or b is None or a == b for a, b in zip(h1, h2))

def minimal_generalization(s, x):
    h = []
    for sv, xv in zip(s, x):
        if sv is None:
            h.append(xv)
        elif sv == xv:
            h.append(sv)
        else:
            h.append("?")
    return tuple(h)

def minimal_specializations(g, x, domains):
    result = []
    for i, gv in enumerate(g):
        if gv == "?":
            for value in domains[i]:
                if value != x[i]:
                    h = list(g)
                    h[i] = value
                    result.append(tuple(h))
    return result

def prune_s(S):
    return {s for s in S if not any(s != s2 and more_general_or_equal(s, s2) for s2 in S)}

def prune_g(G):
    return {g for g in G if not any(g != g2 and more_general_or_equal(g2, g) for g2 in G)}

def display(h):
    return tuple("Ø" if v is None else v for v in h)

df = pd.read_csv("enjoy_sport.csv")
attributes = list(df.columns[:-1])
target = df.columns[-1]
domains = [sorted(df[a].dropna().unique().tolist()) for a in attributes]

S = {tuple([None] * len(attributes))}
G = {tuple(["?"] * len(attributes))}

for idx, row in df.iterrows():
    x = tuple(row[a] for a in attributes)
    positive = str(row[target]).strip().lower() == "yes"

    if positive:
        G = {g for g in G if covers(g, x)}
        new_S = set()
        for s in S:
            if covers(s, x):
                new_S.add(s)
            else:
                h = minimal_generalization(s, x)
                if any(more_general_or_equal(g, h) for g in G):
                    new_S.add(h)
        S = prune_s(new_S)
    else:
        S = {s for s in S if not covers(s, x)}
        new_G = set()
        for g in G:
            if covers(g, x):
                for h in minimal_specializations(g, x, domains):
                    if any(more_general_or_equal(h, s) for s in S):
                        new_G.add(h)
            else:
                new_G.add(g)
        G = prune_g(new_G)

    print(f"Example {idx + 1} ({row[target]}):")
    print("S =", sorted([display(h) for h in S], key=str))
    print("G =", sorted([display(h) for h in G], key=str))
    print()

print("Final S boundary:", sorted([display(h) for h in S], key=str))
print("Final G boundary:", sorted([display(h) for h in G], key=str))

if S == G and len(S) == 1:
    print("Version Space =", {display(next(iter(S)))})


In [ ]:
#Q5
import pandas as pd

def covers(h, x):
    return all(hv is not None and (hv == "?" or hv == xv) for hv, xv in zip(h, x))

def more_general_or_equal(h1, h2):
    return all(a == "?" or b is None or a == b for a, b in zip(h1, h2))

def minimal_generalization(s, x):
    h = []
    for sv, xv in zip(s, x):
        if sv is None:
            h.append(xv)
        elif sv == xv:
            h.append(sv)
        else:
            h.append("?")
    return tuple(h)

def minimal_specializations(g, x, domains):
    result = []
    for i, gv in enumerate(g):
        if gv == "?":
            for value in domains[i]:
                if value != x[i]:
                    h = list(g)
                    h[i] = value
                    result.append(tuple(h))
    return result

def prune_s(S):
    return {s for s in S if not any(s != s2 and more_general_or_equal(s, s2) for s2 in S)}

def prune_g(G):
    return {g for g in G if not any(g != g2 and more_general_or_equal(g2, g) for g2 in G)}

def display(h):
    return tuple("Ø" if v is None else v for v in h)

df = pd.read_csv("enjoy_sport.csv")
attributes = list(df.columns[:-1])
target = df.columns[-1]
domains = [sorted(df[a].dropna().unique().tolist()) for a in attributes]

S = {tuple([None] * len(attributes))}
G = {tuple(["?"] * len(attributes))}

for idx, row in df.iterrows():
    x = tuple(row[a] for a in attributes)
    positive = str(row[target]).strip().lower() == "yes"

    if positive:
        G = {g for g in G if covers(g, x)}
        new_S = set()
        for s in S:
            if covers(s, x):
                new_S.add(s)
            else:
                h = minimal_generalization(s, x)
                if any(more_general_or_equal(g, h) for g in G):
                    new_S.add(h)
        S = prune_s(new_S)
    else:
        S = {s for s in S if not covers(s, x)}
        new_G = set()
        for g in G:
            if covers(g, x):
                for h in minimal_specializations(g, x, domains):
                    if any(more_general_or_equal(h, s) for s in S):
                        new_G.add(h)
            else:
                new_G.add(g)
        G = prune_g(new_G)

    print(f"Example {idx + 1} ({row[target]}):")
    print("S =", sorted([display(h) for h in S], key=str))
    print("G =", sorted([display(h) for h in G], key=str))
    print()

print("Final S boundary:", sorted([display(h) for h in S], key=str))
print("Final G boundary:", sorted([display(h) for h in G], key=str))

if S == G and len(S) == 1:
    print("Version Space =", {display(next(iter(S)))})
